<a href="https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Connected.


## 1. Unit of analysis + time window

One row = one content item, on one day (content_hash_id x report_date), within a mid-panel month (month=2026-03) — I'm avoiding the final month (June 2026) since it's a sealed test window, not for iterating on label logic. My lane needs this daily grain because "declining" is a change over time, which only exists at the daily level, not a single snapshot.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Feature (knowable before the decision point): gsc_impressions, gsc_clicks, gsc_avg_position — from the PRIOR window only.
Label/proxy: whether impressions decline in the LATER window (I compute this myself, not from a pre-built flag).
Context (grouping/joining only, never a feature): client_hash_id, content_hash_id, report_date.
Excluded: any FlyRank product decision fields (health_score, priority_score, action_type) — these aren't in this dataset at all, which is intentional, so there's nothing to exclude by hand; also excluding raw query/URL/title fields since none are shipped in the pseudonymized release.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

Three checks below: (a) the grain really is one row per content item per day, (b) row count and date span for March 2026, (c) availability of GSC data using IS TRUE filtering.

In [3]:
# (a) Grain check — should return 0 rows if grain holds
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) c
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY 1, 2
    HAVING c > 1
    LIMIT 5
""").df()
print("Grain violations (should be empty):")
print(grain_check)

# (b) Row count + date span for the month
counts = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
print("\nRow count + date span:")
print(counts)

# (c) Availability — GSC data flagged available
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
print("\nAvailability:")
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be empty):
Empty DataFrame
Columns: [content_hash_id, report_date, c]
Index: []

Row count + date span:
   row_count      min_d      max_d
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability:
   total_rows  gsc_available_rows
0     9841378           3611061.0


## 4. Data limits

One named limitation: this is an unbalanced panel — different clients have different history lengths, and only ~37% of March rows (3.61M of 9.84M) have GSC data flagged as available. Rows without availability likely reflect tracking not yet started for that client, not zero traffic — so I need to filter on availability before treating missing values as "no visibility."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
# Five features, each knowable BEFORE the decision moment
feat = con.sql(f"""
    SELECT content_hash_id,
           AVG(gsc_impressions) AS avg_impressions,
           AVG(gsc_clicks) AS avg_clicks,
           AVG(gsc_avg_position) AS avg_position,
           COUNT(*) AS days_with_data,
           MAX(gsc_impressions) AS peak_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY content_hash_id
""").df()
feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,avg_impressions,avg_clicks,avg_position,days_with_data,peak_impressions
0,content_b7e512995f79d5a6,28.600000,0.133333,4.247255,15,43
1,content_05597932fe4da067,1.200000,0.000000,4.939394,15,3
2,content_905aa32a0230694e,5.933333,0.000000,3.010741,15,10
3,content_05434271b257bb68,41.866667,0.066667,5.330069,15,62
4,content_d056587ff7faca0c,85.333333,0.600000,4.468441,15,178


avg_impressions — known: it's just an average of past daily impressions.
avg_clicks — known: same, from the past window only.
avg_position — known: search position observed in the past window.
days_with_data — known: a count of past rows, no future info.
peak_impressions — known: max value seen in the past window only.

In [5]:
# THE TRAP — add a label-derived column on purpose, watch the score jump
import numpy as np
feat_leaky = feat.copy()
feat_leaky["is_declining"] = np.random.randint(0, 2, len(feat_leaky))  # placeholder label for demo
feat_leaky["leaky_col"] = feat_leaky["is_declining"]  # <-- THIS is literally the label copied in

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_leaky = feat_leaky[["avg_impressions", "leaky_col"]].fillna(0)
y = feat_leaky["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X_leaky, y, test_size=0.25, random_state=42)
model = LogisticRegression().fit(X_tr, y_tr)
print("Score WITH leaky column:", model.score(X_te, y_te))

# Now remove it and check the honest score
X_honest = feat_leaky[["avg_impressions"]].fillna(0)
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42)
model_honest = LogisticRegression().fit(X_tr, y_tr)
print("Score WITHOUT leaky column (honest):", model_honest.score(X_te, y_te))

Score WITH leaky column: 1.0
Score WITHOUT leaky column (honest): 0.5011508631473606


As expected, adding a column that's literally derived from the label pushed accuracy to 1.0 — a dead giveaway of leakage. Removing it dropped the score back to ~0.50 (near chance, since this label was random noise for the demo). This is the check I'll run for real once I define my actual target: if any single feature alone gets suspiciously close to a perfect score, I'll assume it's leaking the answer and investigate.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.